# Stage 2 & 3: Isolation Forest Training and Validation
**Motor Predictive Maintenance - IoT Induction Motor Project**

In this notebook, we train Isolation Forest strictly on normal data, validate false alarm rates on held-out normal data, and benchmark against the statistical baseline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, "../ml")
from data_loader import load_normal_data, SELECTED_FEATURES
from baseline_model import StatisticalDistanceBaseline
from train_anomaly_model import build_and_train_isolation_forest, evaluate_on_validation_normal

data_dir = Path("../data/raw/SKAB")
normal_csv = data_dir / "anomaly-free/anomaly-free.csv"
df_train, df_val = load_normal_data(normal_csv, train_ratio=0.8)
print(f"Train samples (Normal): {len(df_train)}")
print(f"Validation samples (Normal): {len(df_val)}")


## 1. Fit Preprocessing and Models Strictly on Train Split

In [ ]:
X_train = df_train[SELECTED_FEATURES]
X_val = df_val[SELECTED_FEATURES]

# Fit Isolation Forest pipeline (StandardScaler + IsolationForest)
iso_pipeline = build_and_train_isolation_forest(X_train, contamination=0.01, random_state=42)

# Fit Statistical Baseline (Mahalanobis Distance)
baseline = StatisticalDistanceBaseline(alpha=0.01)
baseline.fit(X_train)
print("Both models fitted strictly on training observations.")


## 2. Calibrate False Alarm Rate on Unseen Normal Validation Set

In [ ]:
val_metrics = evaluate_on_validation_normal(iso_pipeline, baseline, X_val)
print("Validation Results (False Alarm Rate on Healthy Unseen Data):")
for k, v in val_metrics.items():
    print(f"  {k}: {v}")


## 3. Run Benchmark Evaluation on Held-Out Test Set

In [ ]:
from evaluate_anomaly_model import run_full_evaluation

model_dir = Path("../ml/saved_models")
reports_dir = Path("../reports")
results_df = run_full_evaluation(model_dir, data_dir, reports_dir)
display(results_df)
